In [95]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE



In [69]:
urinalysis_data = pd.read_csv(r"datasets/urinalysis_data_cleaned.csv")

In [70]:
urinalysis_data.head()

,Age,Gender,Color,Transparency,Glucose,Protein,pH,Specific Gravity,WBC,RBC,Epithelial Cells,Mucous Threads,Amorphous Urates,Bacteria,Diagnosis
0,76.0,FEMALE,LIGHT YELLOW,CLEAR,NEGATIVE,NEGATIVE,5.0,1.010,1-3,0-2,OCCASIONAL,RARE,NONE SEEN,OCCASIONAL,NEGATIVE
1,9.0,MALE,DARK YELLOW,SLIGHTLY HAZY,NEGATIVE,1+,5.0,1.030,1-3,0-2,RARE,FEW,FEW,MODERATE,NEGATIVE
2,12.0,MALE,LIGHT YELLOW,SLIGHTLY HAZY,NEGATIVE,TRACE,5.0,1.030,0-3,0-2,RARE,FEW,MODERATE,RARE,NEGATIVE
3,77.0,MALE,BROWN,CLOUDY,NEGATIVE,1+,6.0,1.020,5-8,LOADED,RARE,RARE,NONE SEEN,FEW,NEGATIVE
4,29.0,FEMALE,YELLOW,HAZY,NEGATIVE,TRACE,6.0,1.025,1-4,0-2,RARE,RARE,NONE SEEN,FEW,NEGATIVE


In [71]:
urinalysis_data.isna().sum()

Age                 0
Gender              0
Color               0
Transparency        0
Glucose             0
Protein             0
pH                  0
Specific Gravity    0
WBC                 0
RBC                 0
Epithelial Cells    0
Mucous Threads      0
Amorphous Urates    0
Bacteria            0
Diagnosis           0
dtype: int64

## Feature Engineering

In [72]:
X  = urinalysis_data.drop(columns=["Diagnosis"])
Y = urinalysis_data["Diagnosis"]

## Train-Test split

In [73]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=32)

## Feature Encoding

In [74]:
# select categrorical features
categorical_features = x_train.select_dtypes(include=["object"]).columns.tolist()

In [75]:
# create instance
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# Fit on training data
encoder.fit(x_train[categorical_features])

# Transform datasets
x_train[categorical_features] = encoder.transform(x_train[categorical_features])
x_test[categorical_features] = encoder.transform(x_test[categorical_features])

In [76]:
# Encode target variable
y_train = y_train.replace({"NEGATIVE": 0, "POSITIVE": 1})
y_test = y_test.replace({"NEGATIVE": 0, "POSITIVE": 1})

C:\Users\user\AppData\Local\Temp\ipykernel_11900\3283856114.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_train = y_train.replace({"NEGATIVE": 0, "POSITIVE": 1})
C:\Users\user\AppData\Local\Temp\ipykernel_11900\3283856114.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_test = y_test.replace({"NEGATIVE": 0, "POSITIVE": 1})


## Apply SMOTE

In [77]:
smote = SMOTE(random_state=42)

x_retrained, y_retrained = smote.fit_resample(x_train, y_train)

## Feature Scaling

In [78]:
# create instance of scaler
scaler = StandardScaler()

# scale datasets
x_retrained_scaled = scaler.fit_transform(x_retrained)
x_test_scaled = scaler.transform(x_test)

## Train Models
- Logistic Regression (baseline model)
- Random Forest Classifier
- Decision Tree Classifier
- XGBoost Classifier

In [79]:
# create instances of models
logistic_model = LogisticRegression(random_state=32)
decision_tree_model = DecisionTreeClassifier(random_state=32)
random_forest_model = RandomForestClassifier(random_state=32)
xgboost_model = XGBClassifier(random_state=32)

In [80]:

# Perform cross-validation for each model

logistic_scores = cross_val_score(logistic_model, x_retrained_scaled, y_retrained, cv=5, scoring='accuracy')
decision_tree_scores = cross_val_score(decision_tree_model, x_retrained_scaled, y_retrained, cv=5, scoring='accuracy')
random_forest_scores = cross_val_score(random_forest_model, x_retrained_scaled, y_retrained, cv=5, scoring='accuracy')
xgboost_scores = cross_val_score(xgboost_model, x_retrained_scaled, y_retrained, cv=5, scoring='accuracy')

# Print average scores
print(f"Logistic Regression Accuracy: {logistic_scores.mean():.4f}")
print(f"Decision Tree Accuracy: {decision_tree_scores.mean():.4f}")
print(f"Random Forest Accuracy: {random_forest_scores.mean():.4f}")
print(f"XGBoost Accuracy: {xgboost_scores.mean():.4f}")

Logistic Regression Accuracy: 0.7202
Decision Tree Accuracy: 0.9372
Random Forest Accuracy: 0.9714
XGBoost Accuracy: 0.9650


In [81]:
# fit models with data
logistic_model.fit(x_retrained_scaled, y_retrained)
decision_tree_model.fit(x_retrained_scaled, y_retrained)
random_forest_model.fit(x_retrained_scaled, y_retrained)
xgboost_model.fit(x_retrained_scaled, y_retrained)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [82]:
# model prediction
logistic_predictions = logistic_model.predict(x_test_scaled)
decision_tree_predictions = decision_tree_model.predict(x_test_scaled)
random_forest_predictions = random_forest_model.predict(x_test_scaled)
xgboost_predictions = xgboost_model.predict(x_test_scaled)

## Model Evaluation

In [87]:
# model evaluation (accuracy score)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, logistic_predictions):.4f}")
print(f"Decision Tree Accuracy: {accuracy_score(y_test, decision_tree_predictions):.4f}")
print(f"Random Forest Accuracy: {accuracy_score(y_test, random_forest_predictions):.4f}")
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgboost_predictions):.4f}")

Logistic Regression Accuracy: 0.7944
Decision Tree Accuracy: 0.9129
Random Forest Accuracy: 0.9512
XGBoost Accuracy: 0.9547


In [92]:
print(classification_report(y_test, logistic_predictions, target_names=["NEGATIVE", "POSITIVE"]))
print("____________________________________________________________________")
print(classification_report(y_test, decision_tree_predictions, target_names=["NEGATIVE", "POSITIVE"]))
print("____________________________________________________________________")
print(classification_report(y_test, random_forest_predictions, target_names=["NEGATIVE", "POSITIVE"]))
print("____________________________________________________________________")
print(classification_report(y_test, xgboost_predictions, target_names=["NEGATIVE", "POSITIVE"]))

              precision    recall  f1-score   support

    NEGATIVE       0.97      0.81      0.88       271
    POSITIVE       0.15      0.56      0.23        16

    accuracy                           0.79       287
   macro avg       0.56      0.69      0.56       287
weighted avg       0.92      0.79      0.85       287

____________________________________________________________________
              precision    recall  f1-score   support

    NEGATIVE       0.96      0.95      0.95       271
    POSITIVE       0.26      0.31      0.29        16

    accuracy                           0.91       287
   macro avg       0.61      0.63      0.62       287
weighted avg       0.92      0.91      0.92       287

____________________________________________________________________
              precision    recall  f1-score   support

    NEGATIVE       0.95      1.00      0.97       271
    POSITIVE       0.75      0.19      0.30        16

    accuracy                           0.95 

In [96]:
logistic_auc_score = roc_auc_score(y_test, logistic_predictions)
decision_tree_auc_score = roc_auc_score(y_test, decision_tree_predictions)
random_forest_auc_score = roc_auc_score(y_test, random_forest_predictions)
xgboost_auc_score = roc_auc_score(y_test, xgboost_predictions)

print(f"Logistic Regression AUC-ROC: {logistic_auc_score:.4f}")
print(f"Decision Tree AUC-ROC: {decision_tree_auc_score:.4f}")
print(f"Random Forest AUC-ROC: {random_forest_auc_score:.4f}")
print(f"XGBoost AUC-ROC: {xgboost_auc_score:.4f}")


Logistic Regression AUC-ROC: 0.6853
Decision Tree AUC-ROC: 0.6304
Random Forest AUC-ROC: 0.5919
XGBoost AUC-ROC: 0.6232


## Model Optimization / Fine tuning